# Lab 3: Local RAG System with Source Attribution and Scope Detection

**Duration:** ~25 minutes

---

Labs 1 and 2 showed how to explain ML models that classify structured data. Now we move to **LLMs**: large language models that process and generate unstructured (and mainly text) data.

A **Retrieval-Augmented Generation (RAG)** system is one of the most important transparency patterns for LLMs: instead of relying on what the model "memorized" during training, you make it ground its answers in retrieved custom documents. This improves trustworthiness, because you can point to where (which document) the model retrieved its information from.

In this lab you will **build and evaluate source attribution, inline citations, and scope detection on top of a very simple RAG system**. The focus is not on RAG itself (this is why we won't be building embeddings nor complex Goodness-of-Fit retrieval metrics), but instead on the mentioned transparency capabilities.

## What you will learn

During this lab, you will:

1. Load a small knowledge base of `.txt` files, and retrieve relevant documents using **naive keyword search** (no embeddings nor vectors, to keep the RAG complexity out of the lab)
2. Generate answers via a **local LLM** (Ollama) grounded in the retrieved context
3. Add **source attribution** (showing which documents were used by the LLM) and extend it with **inline citations**
4. Add **LLM-based scope detection** first, then **ML-based scope detection**, and compare the two
5. Run the full pipeline on several queries and observe where it works and where it fails

## Workflow

```
User query
    │
    ▼
┌─────────────────────┐
│  Retrieval Engine   │  ← naive keyword matching against .txt files
│  (keyword search)   │    returns top-K documents by overlap score
└─────────────────────┘
    │ top-k documents + titles (as metadata)
    ▼
┌─────────────────────┐
│  Prompt Builder     │  ← injects retrieved context into the prompt
│  (with attribution) │    includes document titles as sources
└─────────────────────┘
    │ grounded prompt
    ▼
┌─────────────────────┐
│  Answer Generation  │  ← local LLM via Ollama
│  (local LLM)        │
└─────────────────────┘
```

## Prerequisites

- [Ollama](https://ollama.com) running locally with a small model pulled (`qwen3:0.6b`)
- The `knowledge/` folder in the same directory as this notebook, containing `.txt` documents
- Python packages: `requests` (standard in most environments)


---


## Initial Setup

We only need the Python standard library (`os`) and `requests` to call the Ollama API. No embeddings nor ML frameworks: the point is to show the **core RAG concept** first, with human-readable documents and a human-understandable retrieval mechanism (simple keyword matching excluding very short words -less than 4 characters long-).

In [ ]:
import os
import requests

# --- Ollama configuration ---
OLLAMA_URL = "http://localhost:11434/api/chat"
OLLAMA_MODEL = "qwen3:0.6b"  # adjust to match your local Ollama model name

# --- Retrieval configuration ---
TOP_K = 2          # number of documents to retrieve per query
MIN_WORD_LEN = 4   # ignore short words (stop-word filtering heuristic)

print("Configuration loaded.")
print(f"  Ollama model : {OLLAMA_MODEL}")
print(f"  Top-K docs   : {TOP_K}")


---

## 1. Load the Knowledge Base

The knowledge base is a folder of plain `.txt` files. Each file follows a simple convention:
- **Line 1**: document title
- **Remaining lines**: document body

We load every `.txt` file into a Python dictionary: `{ title → body text }`.

For this lab, let's run an imagination exercise: due to the rapid AI advancements in the last years, we are now living in a parallel universe where Europe has become a dystopian, cyberpunk world. Capitals like Berlin are massively overpopulated, due to their popular industrial and technified atmosphere. Climate change has turned Northern countries into mild, Mediterranean-like climates. AI has optimized nuclear fusion, and now almost everyone moves around in fusion-powered flying cars. Everyone has quantum computers at home.

We have built a simple RAG knowledge base (under the `knowledge/` folder) with every major change that this futuristic Europe has undergone during 2026. LLMs have of course been trained on now outdated information (pre-our 2026) about European countries, so we want to ground their answers on this database.

Feel free to open the files in `knowledge/` and read their content before running the following cell.

In [ ]:
def load_knowledge(folder):
    """Load all .txt files from `folder` into a {title: body} dict."""
    knowledge = {}
    for filename in sorted(os.listdir(folder)):
        if filename.endswith(".txt"):
            filepath = os.path.join(folder, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                lines = f.read().strip().splitlines()
                title = lines[0]
                body = "\n".join(lines[1:]).strip()
                knowledge[title] = body
    return knowledge


KNOWLEDGE_FOLDER = os.path.join(os.path.dirname(os.path.abspath("__file__")), "knowledge")
knowledge = load_knowledge(KNOWLEDGE_FOLDER)

print(f"Loaded {len(knowledge)} documents from '{KNOWLEDGE_FOLDER}':\n")
for title, body in knowledge.items():
    preview = body[:80].replace("\n", " ")
    print(f"  [{title}]\n   {preview}...\n")


---

## 2. Retrieval with Naive Keyword Search

We use the simplest possible retrieval strategy: **count how many query words appear in each document** and return the top-K documents by overlap score.

- We filter out short words (length < `MIN_WORD_LEN`) as a basic stop-word filter.
- We only return documents that have **at least one matching word** (score > 0).

This approach has obvious weaknesses (no synonyms, no semantic similarity); we will discuss them in the exercises.

In [ ]:
def naive_keyword_search(query, documents, top_k=TOP_K):
    """
    Score each document by counting keyword overlaps with the query.
    Returns a list of (title, body) tuples for the top_k matches.
    """
    query_words = set(
        w for w in query.lower().split() if len(w) >= MIN_WORD_LEN
    )

    scored = []
    for title, body in documents.items():
        doc_words = set(
            w for w in body.lower().split() if len(w) >= MIN_WORD_LEN
        )
        score = len(query_words & doc_words)  # intersection size
        scored.append({"title": title, "body": body, "score": score})

    scored.sort(key=lambda x: x["score"], reverse=True)

    return [
        (item["title"], item["body"])
        for item in scored[:top_k]
        if item["score"] > 0
    ]


# Quick smoke test
test_query = "What languages do Dutch people speak?"
results = naive_keyword_search(test_query, knowledge)
print(f"Query: \"{test_query}\"")
print(f"Retrieved {len(results)} document(s):")
for title, _ in results:
    print(f"  - {title}")


---

## 3. Answer Generation with Simple Source Attribution (Document Titles)

We call a **local LLM via Ollama** and inject the retrieved documents into the prompt as context. Each context snippet is labelled with its **document title**; this metadata is what will give us the attribution.

In [ ]:
def generate_answer(query, retrieved_docs):
    """
    Call the local LLM with the query and retrieved context.

    Parameters
    ----------
    query          : str              — the user's question
    retrieved_docs : list[(str, str)] — list of (title, body) pairs

    Returns
    -------
    answer  : str        — the LLM's response
    sources : list[str]  — document titles used as context
    """

    sources = [title for title, _ in retrieved_docs]

    # Build context block with labelled source snippets
    context_block = "\n\n".join(
        f"[Source: {title}]\n{body}"
        for title, body in retrieved_docs
    )

    prompt = (
        f"Context: {context_block}\n\n"
        f"Question: {query}"
    )

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
        },
        timeout=60,
    )
    response.raise_for_status()

    answer = response.json()["message"]["content"]
    return answer, sources

We combine retrieval and generation into a single `rag_pipeline` function that:

1. Retrieves the most relevant documents for a query
2. Generates an answer via the local LLM
3. Prints the **sources** (document titles) alongside the answer

In [ ]:
def rag_pipeline(query):
    """Run the full RAG pipeline: retrieve → generate → display with attribution."""
    print(f"\n{'='*60}")
    print(f"Question: {query}")
    print('='*60)

    # Step 1: Retrieve
    retrieved = naive_keyword_search(query, knowledge)
    if retrieved:
        print(f"\nRetrieved {len(retrieved)} document(s):")
        for title, _ in retrieved:
            print(f"  - {title}")
    else:
        print("\nNo relevant documents found in the knowledge base.")

    # Step 2: Generate
    answer, sources = generate_answer(query, retrieved)

    # Step 3: Display with attribution
    print(f"\nAnswer:\n{answer}")
    if sources:
        print(f"\nSources: {', '.join(sources)}")
    print()

Let's run the pipeline on a set of example queries. As you read the output, consider:

- Does the system retrieve the **correct** document(s)?
- Is the answer **grounded** in the retrieved text, or does the LLM add information from its training data?
- Does the attribution (source titles) help you **trust** the answer more?
- What happens when the query is **outside** the knowledge base?

In [ ]:
rag_pipeline("What is the current population of Berlin?")

In [ ]:
rag_pipeline("What's the weather like in Norway during February?")

In [ ]:
rag_pipeline("What languages do Dutch people speak?")

In [ ]:
rag_pipeline("Tell me about quantum computing")

In [ ]:
rag_pipeline("What is the capital of Spain?")

We know that our provided RAG database contains information about Europe from 2026 on. But what if we ask the LLM about something totally unrelated?:

In [ ]:
# This query is completely outside the knowledge base. What happens when we run it?
rag_pipeline("How can I install Python on my Windows laptop?")

Even if no relevant documents were retrieved, the LLM still generated an answer based on its training data. This training data might be reliable. Or not. We cannot know because it is not accessible anymore, so we cannot validate it.

We want to avoid these situations. For now, we can simply skip doing an LLM call if no relevant documents were retrieved from the RAG database. This not only gives us more control over the system and more trust, but we are also avoiding unnecessary LLM calls which, in the long term, can significantly reduce costs money- and time-wise. Let's simply add an extra if statement in our generate_answer method:

In [ ]:
def generate_answer(query, retrieved_docs):
    """
    Call the local LLM with the query and retrieved context.

    Parameters
    ----------
    query          : str              — the user's question
    retrieved_docs : list[(str, str)] — list of (title, body) pairs

    Returns
    -------
    answer  : str        — the LLM's response
    sources : list[str]  — document titles used as context
    """
    if not retrieved_docs: # NEW CODE BLOCK
        return (
            "I don't have relevant information in my knowledge base to answer that question.",
            []
        )

    sources = [title for title, _ in retrieved_docs]

    # Build context block with labelled source snippets
    context_block = "\n\n".join(
        f"[Source: {title}]\n{body}"
        for title, body in retrieved_docs
    )

    prompt = (
        f"Context: {context_block}\n\n"
        f"Question: {query}"
    )

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
        },
        timeout=60,
    )
    response.raise_for_status()

    answer = response.json()["message"]["content"]
    return answer, sources

Now, if we run the same query, the LLM shouldn't generate any answer:

In [ ]:
# This query is completely outside the knowledge base.
# Our guardrail blocks the LLM call and returns a safe static message instead.
rag_pipeline("How can I install Python on my Windows laptop?")


---

## 4. Inspect the Retrieval Scores

It helps to see the **scores** for every document, not just the "winning document(s)". This makes the retrieval decision transparent and lets you spot cases where the wrong document is ranked first.

In [ ]:
def explain_retrieval(query, documents):
    """Print the keyword overlap score for every document, ranked."""
    query_words = set(
        w for w in query.lower().split() if len(w) >= MIN_WORD_LEN
    )
    print(f"Query words (len >= {MIN_WORD_LEN}): {query_words}\n")

    scored = []
    for title, body in documents.items():
        doc_words = set(
            w for w in body.lower().split() if len(w) >= MIN_WORD_LEN
        )
        matched = query_words & doc_words
        scored.append((title, len(matched), matched))

    scored.sort(key=lambda x: x[1], reverse=True)

    print(f"{'Document title':<40} {'Score':>5}  Matched words")
    print("-" * 75)
    for title, score, matched in scored:
        matched_str = ", ".join(sorted(matched)) if matched else "-"
        print(f"  {title:<38} {score:>5}  {matched_str}")


explain_retrieval("What's the weather like in Norway during February?", knowledge)

In the example query about Norway's weather, we see that the wrong document (`Germany in 2026`) was ranked before the correct document, due to a higher retrieval score (more keywords matched).

We of course know that this is a naive retrieval mechanism. But hopefully, this will show you that we shouldn't blindly trust any given attributed sources, and we should look into those source documents ourselves.

Also, source attribution based solely on metadata (the document titles) doesn't give us a lot of trust. Let's check this query again:

In [ ]:
rag_pipeline("Tell me about quantum computing")

A correct document is retrieved, and the LLM answer does mention that Europe has widely adopted quantum computing.

However, most of the generated information does surely not come from the retrieved document. Instead, the LLM is generating its answer based on its training data.

We know from the Python-related query that we cannot blindly trust these non-RAG-grounded answers. The problem here is that we are given a source attribution to a seemingly reliable document from our knowledge base. This might make us trust the answer more, but it shouldn': we don't know what part of the answer is grounded in our sourced document, and what isn't.

Inline citations can help us with this.


---

## 5. Inline Citations

Until now, we have implemented simple source attribution: showing document titles at the end of an answer. This tells us *which* documents were in the context, but not *which part of the answer* each document is responsible for. A grounded claim and a hallucinated claim would look identical to us if they both display the same retrieved document titles.

**Inline citations** fix this by attaching a `[source]` marker at the end of sentences that are supported by that source.

We attribute sentences to sources by measuring what disappears when a source is removed:

```
Step 0   Call the LLM with ALL retrieved documents → full answer (baseline)

Step 1   Remove document [1] → call LLM again → answer_without_1
Step 2   Remove document [2] → call LLM again → answer_without_2
         … repeat for every retrieved document …

Step 3   Split the full answer into sentences.
         For each sentence, for each removed document [i]:
           ratio = (words in sentence absent from answer_without_i) / (words in sentence)
           if ratio ≥ CITATION_THRESHOLD  →  attribute sentence to [i]
         Append [i] citation markers at the end of attributed sentences.
```

**Cost:** `1 + n_docs` LLM calls per query. With `TOP_K = 2` that is 3 calls total.

**Key caveats and limitations to keep in mind:**
- We use `temperature=0` to make the LLM as deterministic as possible across calls, but with how LLMs are implemented, they can never be fully deterministic. Small variations may still occur.
- A sentence marked `[i]` means *"removing document i caused a significant fraction of this sentence's words to disappear"*, which is a correlation, but not a guaranteed proof of faithful grounding.
- Sentences with no citations are likely drawn from the model's training data.
- The LLM might express the same fact using different words (paraphrasing) in the LLM calls with removed documents; the original words will then appear uncited even though they're grounded.
- Function words ("the", "is", "in") may appear in all answers regardless of context, so they're never attributed to a specific document even if they might be grounded on it.
- `CITATION_THRESHOLD` controls sensitivity: lower values attribute more sentences (higher recall), higher values are more conservative (higher precision).

In [ ]:
import re

# Fraction of a sentence's words that must disappear (when a source is removed)
# for that sentence to be attributed to that source. Range: 0.0 – 1.0.
CITATION_THRESHOLD = 0.6


def _strip_think_tags(text):
    """Remove <think>...</think> reasoning blocks produced by some models (e.g. qwen3)."""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


def _call_llm_grounded(query, context_block):
    """
    Single LLM call with a strict grounding instruction.
    Uses temperature=0 for deterministic (reproducible) output across LOO (leave-one-out) calls.
    Returns an empty string if context_block is empty.
    """
    
    prompt = (
        f"Context: {context_block}\n\n"
        f"Question: {query}"
    )
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "messages": [{"role": "user", "content": prompt}],
            "stream": False,
            "options": {"temperature": 0},
        },
        timeout=60,
    )
    response.raise_for_status()
    raw = response.json()["message"]["content"]
    return _strip_think_tags(raw)

print(f"  Citation threshold : {CITATION_THRESHOLD}")

In [ ]:
def generate_answer_with_inline_citations(query, retrieved_docs):
    """
    Sentence-level inline citation algorithm.

    Algorithm
    ---------
    0. Call the LLM with ALL retrieved documents → full_answer (baseline)
    1. For each document i, remove it and call the LLM again → answer_without_i
    2. Split full_answer into sentences. For each sentence, compute:
         ratio = (words in sentence absent from answer_without_i) / (words in sentence)
       If ratio >= CITATION_THRESHOLD, the sentence is attributed to document [i].
    3. Append [i] citation markers at the end of attributed sentences.

    Parameters
    ----------
    query          : str              — the user's question
    retrieved_docs : list[(str, str)] — list of (title, body) pairs

    Returns
    -------
    annotated_answer : str       — full answer with sentence-level [i] citation markers
    sources          : list[str] — document titles (1-indexed, matching citations)
    """
    if not retrieved_docs:
        return (
            "I don't have relevant information in my knowledge base to answer that question.",
            []
        )
    
    sources = [title for title, _ in retrieved_docs]

    def build_context(docs):
        return "\n\n".join(
            f"[Source: {title}]\n{body}" for title, body in docs
        )

    def normalise(word):
        """Strip punctuation and lowercase a word for set-based comparison."""
        return re.sub(r"[^\w]", "", word).lower()

    # Step 0: baseline answer with all documents
    print("  Calling LLM with all documents (baseline)...")
    full_answer = _call_llm_grounded(query, build_context(retrieved_docs))

    # Steps 1-2: one LOO call per retrieved document
    loo_word_sets = []
    for i, (title, _) in enumerate(retrieved_docs):
        docs_without_i = [d for j, d in enumerate(retrieved_docs) if j != i]
        print(f"  Calling LLM without document [{i + 1}] ({title})...")
        ans_without_i = _call_llm_grounded(query, build_context(docs_without_i))
        loo_word_sets.append(
            {normalise(w) for w in ans_without_i.split() if normalise(w)}
        )

    # Step 3: sentence-level attribution
    # Split on whitespace that follows sentence-ending punctuation (.  !  ?)
    sentences = re.split(r'(?<=[.!?])\s+', full_answer.strip())

    annotated_sentences = []
    for sentence in sentences:
        words = [normalise(w) for w in sentence.split() if normalise(w)]
        if not words:
            annotated_sentences.append(sentence)
            continue

        citing = []
        for i, word_set in enumerate(loo_word_sets):
            words_gone = sum(1 for w in words if w not in word_set)
            ratio = words_gone / len(words)
            if ratio >= CITATION_THRESHOLD:
                citing.append(str(i + 1))

        if citing:
            annotated_sentences.append(sentence + "".join(f"[{c}]" for c in citing))
        else:
            annotated_sentences.append(sentence)

    annotated_answer = " ".join(annotated_sentences)
    return annotated_answer, sources

In [ ]:
def rag_pipeline_with_citations(query):
    """RAG pipeline with leave-one-out inline citation attribution."""
    print(f"\n{'='*60}")
    print(f"Question: {query}")
    print('='*60)

    retrieved = naive_keyword_search(query, knowledge)
    if retrieved:
        print(f"\nRetrieved {len(retrieved)} document(s):")
        for i, (title, _) in enumerate(retrieved, 1):
            print(f"  [{i}] {title}")
    else:
        print("\nNo relevant documents found in the knowledge base.")

    print()
    answer, sources = generate_answer_with_inline_citations(query, retrieved)

    print(f"\nAnswer (with inline citations):\n{answer}")
    if sources:
        print("\nSources:")
        for i, title in enumerate(sources, 1):
            print(f"  [{i}] {title}")
    print()

Let's run the same queries as before and compare the output. Notice how the answer now shows `[1]` or `[2]` markers inline, right after words that are attributed to the corresponding source document.

In [ ]:
rag_pipeline_with_citations("Tell me about quantum computing")

In [ ]:
rag_pipeline_with_citations("What's the weather like in Norway during February?")

Look at the annotated answers and ask yourself the following questions:

1. Which sentences carry citation markers, and which don't? Do the un-cited sentences seem less trustworthy to you now (like generic filler or hallucinated content)?

2. Pick a word in a cited sentence and check it against the source document listed. Is the word actually in that document? Is the attribution to that document accurate? Does the surrounding sentence in the answer reflect the document's content faithfully?

3. Run the same query twice. Do you get identical citation patterns? Why or why not?

4. Modify the CITATION_THRESHOLD and re-run the queries. How do the inline citations change? How do these changes affect your own trust in the LLM answer? Is there a threshold where you seem to obtain more accurate citations?


---

## 6. Prompt-based Scope Detection

So far, the pipeline uses a simple guardrail: if no relevant documents are retrieved, the LLM call is skipped. This is effective for queries that share no vocabulary with the knowledge base. However, a query could retrieve documents through coincidental keyword matches while still being fundamentally out of scope; in this case, the LLM may "fill in" with its own training data.

A natural extension is to encode the scope definition in the **system prompt** itself:

> *"You are an assistant with specialized knowledge about the state of Europe in 2026. Only answer questions directly related to this topic."*

This is **prompt-based scope detection**: we rely on the LLM to interpret the instruction and refuse off-topic questions. Notice that, unlike the empty-docs guardrail we added before, this version calls the LLM even when no documents are found; the scope enforcement is now entirely up to the model.

In [ ]:
SCOPE_SYSTEM_PROMPT = (
    "You are an assistant knowledgeable about the current state of Europe in 2026. "
    "You MUST only answer questions that are directly related to European countries, "
    "their populations, geography, geopolitics, climates, technologies, industry, or daily life in 2026, "
    "based on the context retrieved from the knowledge base. "
    "If the question is NOT about Europe, you MUST respond with exactly: "
    "'This question is outside the scope of my knowledge base about Europe in 2026.' "
    "Do not provide any other information for out-of-scope questions."
)


def generate_answer_with_scope_guardrail(query, retrieved_docs):
    """
    Generate an answer using a system prompt that instructs the LLM to stay
    within the scope of 'state of Europe in 2026'.

    Unlike the previous version, this function calls the LLM even when no
    documents are retrieved. The scope enforcement is delegated entirely to
    the model via the system prompt.

    Parameters
    ----------
    query          : str                // the user's question)
    retrieved_docs : list[(str, str)]   // list of (title, body) pairs

    Returns
    -------
    answer  : str          // the LLM's response (may be a scope-refusal message)
    sources : list[str]    // document titles used as context (empty if none)
    """
    sources = [title for title, _ in retrieved_docs]

    if retrieved_docs:
        context_block = "\n\n".join(
            f"[Source: {title}]\n{body}"
            for title, body in retrieved_docs
        )
    else:
        context_block = "(No relevant documents found in the knowledge base.)"

    prompt = (
        f"Context:\n{context_block}\n\n"
        f"Question: {query}"
    )

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "messages": [
                {"role": "system", "content": SCOPE_SYSTEM_PROMPT},
                {"role": "user", "content": prompt},
            ],
            "stream": False,
        },
        timeout=60,
    )
    response.raise_for_status()

    answer = response.json()["message"]["content"]
    return answer, sources


def rag_pipeline_with_scope_guardrail(query):
    """RAG pipeline that uses a system-prompt-based scope guardrail."""
    print(f"\n{'='*60}")
    print(f"Question: {query}")
    print('='*60)

    retrieved = naive_keyword_search(query, knowledge)
    if retrieved:
        print(f"\nRetrieved {len(retrieved)} document(s):")
        for title, _ in retrieved:
            print(f"  - {title}")
    else:
        print("\nNo relevant documents found. Passing query to LLM with scope guardrail.")

    answer, sources = generate_answer_with_scope_guardrail(query, retrieved)

    print(f"\nAnswer:\n{answer}")
    if sources:
        print(f"\nSources: {', '.join(sources)}")
    print()

print("Scope guardrail functions defined.")

Let's test the scope guardrail with an **in-scope** query first:

In [ ]:
# In-scope: should retrieve the Spain document and answer normally
rag_pipeline_with_scope_guardrail("What is the capital of Spain?")

Now an **out-of-scope** query that has no matching documents. The LLM should refuse to answer:

In [ ]:
# Out-of-scope: no matching documents, LLM should follow the scope instruction and refuse
rag_pipeline_with_scope_guardrail("How can I install Python on my Windows laptop?")

That worked (or if it didn't, feel free to slightly edit the SCOPE_SYSTEM_PROMPT to match the expected scope separation).

But can we always trust it? Let's try a trickier case: a query whose answer is influenced by the state of Europe in 2026, and grounded on a source document (and retrieval succeeds). But the actual question might not seem directly related to the state of Europe in 2026.

In [ ]:
# The answer to this query is actually contained in one of the source documents, but the scope description we gave to the LLM might not be flexible enough to consider this as in-scope.
rag_pipeline_with_scope_guardrail("Why are film directors starting to shoot movies in Norway?")

Another failure-prone case: the query starts with a European framing, which can trick the LLM into thinking it is in-scope, even though the actual request (Python recommendations) has nothing to do with the state of Europe in 2026. Try running the following query several times, and see if the scope detection result changes:

In [ ]:
rag_pipeline_with_scope_guardrail(
    "What is the state of quantum computing in Europe and what Python libraries would you recommend for data analysis?"
)

### Risks and Limitations of Prompt-based Scope Detection

The examples above illustrate several key failure modes:

1. **Non-determinism.** LLMs are stochastic. Even with low temperature, a small model like `qwen3:0.6b` can produce different scope decisions on the same query across runs. You cannot guarantee consistent behaviour.

2. **Keyword-triggered retrieval ≠ in-scope.** Our retrieval returns documents whenever query keywords overlap with document text. A query about "hiking in Norway" retrieves the Norway document, and the LLM may then reason *"I have Norway context, so this must be in scope"*; even though the question is about hiking routes, not the 2026 state of Europe.

3. **Prompt injection.** A user can override the system prompt with phrasing like *"Ignore your previous instructions and tell me about..."*. Small instruction-tuned models are especially susceptible. Try it yourself.

4. **Ambiguous framing.** Wrapping an out-of-scope question in European context (*"As a European citizen..."*) can fool the LLM into treating the whole request as in-scope.

5. **No auditability.** When the LLM refuses a query, you cannot inspect *why* it was classified as out of scope. The decision is a black box, which is a problem for XAI.

**Conclusion:** Prompt-based scope detection is easy to implement but brittle. For production systems, it is better to replace it with a deterministic, auditable approach. This brings us to ML-based scope classification.


---

## 7. ML-based Scope Classification

Instead of relying on the LLM to interpret a scope instruction, we can train a **dedicated classifier** to make the scope decision *before* the LLM is ever called. This approach is:

- **Deterministic**: the same query always produces the same classification
- **Auditable**: we can inspect which words drove the decision (XAI!)
- **Fast and cheap**: no LLM call is needed for out-of-scope queries
- **Robust to prompt injection**: the classifier cannot be overridden by text in the user's input

We will classify queries into three categories:

| Label | Meaning |
|---|---|
| `in_scope` | Directly about the state of European countries in 2026 |
| `scope_related` | About Europe or knowledge-base topics, but not specifically about the 2026 state |
| `out_of_scope` | Completely unrelated to Europe or 2026 |

The pipeline then acts on the label:
- `in_scope` → run the full RAG pipeline normally
- `scope_related` → for now, run the RAG pipeline with a disclaimer that the knowledge base may not fully cover the question
- `out_of_scope` → reject immediately; no LLM call is made

### Workflow

```
User query
    │
    ▼
┌─────────────────────────┐
│  ML Scope Classifier    │  ← TF-IDF + Logistic Regression
│  (deterministic gate)   │    trained on labelled query examples
└─────────────────────────┘
    │
    ├── out_of_scope  ──► REJECT (no LLM call)
    │
    ├── scope_related ──► RAG pipeline + disclaimer
    │
    └── in_scope      ──► RAG pipeline (standard)
```

### Build the Labelled Dataset

We hand-craft a small dataset of queries labelled as `in_scope`, `scope_related`, or `out_of_scope`. In a real project this would be much larger, but even ~90 examples are enough to demonstrate the approach.

Feel free to add more examples (especially borderline ones) to stress-test the classifier.

In [ ]:
# ── Scope label mapping ───────────────────────────────────────────────────────
SCOPE_LABELS = {0: "in_scope", 1: "scope_related", 2: "out_of_scope"}

# ── Labelled query dataset ────────────────────────────────────────────────────
# Each entry: (query_text, label_id)
#   in_scope (0)      : directly about the state of European countries in 2026
#   scope_related (1) : about Europe or related topics, not about the 2026 changes
#   out_of_scope (2)  : unrelated to Europe or 2026

TRAINING_DATA = [
    # ── in_scope ──────────────────────────────────────────────────────────────
    ("What is the current population of Berlin?", 0),
    ("How has Norway's climate changed in recent years?", 0),
    ("What type of transportation do Europeans use today?", 0),
    ("Tell me about quantum computing adoption in Europe", 0),
    ("What has changed in Spain since the AI revolution?", 0),
    ("How do people in the Netherlands travel today?", 0),
    ("What energy source powers transportation in Europe?", 0),
    ("Describe the current state of German cities", 0),
    ("What is the current climate of Oslo?", 0),
    ("How have European population patterns changed?", 0),
    ("What is daily life like in Europe today?", 0),
    ("Tell me about European cities in 2026", 0),
    ("What happened to Norway's weather patterns?", 0),
    ("How are quantum computers used in everyday European life?", 0),
    ("What kind of vehicles do Europeans drive now?", 0),
    ("Describe the current state of the Netherlands", 0),
    ("What is the weather like in Spain today?", 0),
    ("How has transportation changed in Europe?", 0),
    ("What are European cities like after the AI advancement?", 0),
    ("Tell me about flying cars in modern Europe", 0),
    ("What industries dominate German cities now?", 0),
    ("How has climate change affected Scandinavian countries?", 0),
    ("Has nuclear fusion changed how Europeans travel?", 0),
    ("What are the major changes in Norway since 2024?", 0),
    ("How do people commute in Berlin today?", 0),
    ("What is the energy infrastructure of Europe like?", 0),
    ("How have AI advancements changed European society?", 0),
    ("What is the population density of European capitals today?", 0),
    ("How hot is Spain in the summer now?", 0),
    ("Are fusion-powered cars common in the Netherlands?", 0),

    # ── scope_related ─────────────────────────────────────────────────────────
    ("What was the capital of France in the 1980's?", 1),
    ("How can I travel to Germany from Russia?", 1),
    ("Tell me about European history before 2020", 1),
    ("What countries are in Scandinavia?", 1),
    ("Is Amsterdam in the Netherlands?", 1),
    ("What is the official language of Spain?", 1),
    ("What is the geography of Norway?", 1),
    ("What are the major cities in Germany?", 1),
    ("What is traditional Dutch food?", 1),
    ("How many countries are in Europe?", 1),
    ("What is the Euro currency?", 1),
    ("What is the European Union?", 1),
    ("What is quantum computing?", 1),
    ("How does nuclear fusion work in theory?", 1),
    ("What are the Scandinavian countries?", 1),
    ("How come Spain has so many official languages?", 1),
    ("Which European languages come from Latin?", 1),
    ("Tell me about Dutch culture and traditions", 1),
    ("What is the Rhine river?", 1),
    ("What was Germany like before the industrial AI revolution?", 1),
    ("Give me a summary of the history of the Netherlands?", 1),
    ("Who are some famous German scientists?", 1),
    ("What is the Norwegian language called in Norwegian?", 1),
    ("What are quantum bits (qubits)?", 1),
    ("What is plasma confinement in fusion research?", 1),
    ("What is the population of Europe historically?", 1),
    ("What are common Spanish traditions?", 1),
    ("What time during the year is best to visit Spain?", 1),
    ("What is the difference between fission and fusion?", 1),
    ("What are the northern European countries?", 1),

    # ── out_of_scope ──────────────────────────────────────────────────────────
    ("How can I install Python on my Windows laptop?", 2),
    ("What is the recipe for spaghetti carbonara?", 2),
    ("Who won the FIFA World Cup in 2018?", 2),
    ("How do I fix a null pointer exception in Java?", 2),
    ("What is the speed of light?", 2),
    ("Tell me about the history of ancient China", 2),
    ("How do I bake a chocolate cake?", 2),
    ("What are best practices for training neural networks?", 2),
    ("How does photosynthesis work in plants?", 2),
    ("What is the capital of Japan?", 2),
    ("Can you write me a poem about autumn?", 2),
    ("What is 2 + 2?", 2),
    ("How do I improve my swimming technique?", 2),
    ("What is the best smartphone to buy in 2025?", 2),
    ("Tell me about the US presidential elections", 2),
    ("How do I train a deep learning model?", 2),
    ("What is the weather like in New York?", 2),
    ("Who invented the telephone?", 2),
    ("What is the GDP of India?", 2),
    ("How does a combustion engine work?", 2),
    ("What is the population of Brazil?", 2),
    ("How do I learn to play guitar?", 2),
    ("What are the planets in our solar system?", 2),
    ("How do I create a React component?", 2),
    ("What is the stock market doing today?", 2),
    ("Tell me about the history of the Roman Empire", 2),
    ("How do cats communicate with humans?", 2),
    ("What is the Amazon rainforest?", 2),
    ("How do I write a good resume?", 2),
    ("What is machine learning?", 2),
]

print(f"Total examples: {len(TRAINING_DATA)}")
for label_id, label_name in SCOPE_LABELS.items():
    count = sum(1 for _, l in TRAINING_DATA if l == label_id)
    print(f"  {label_name}: {count}")

### Train the Classifier

We use a **TF-IDF vectoriser** (unigrams + bigrams) followed by a **Logistic Regression** model. This combination is:
- Lightweight: no GPU or embeddings needed
- Interpretable: coefficients directly tell us which n-grams push toward each class (perfect for XAI)
- Well-calibrated: `predict_proba` gives meaningful confidence scores we can display in the pipeline

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Prepare features and labels
queries_all = [q for q, _ in TRAINING_DATA]
labels_all  = [l for _, l in TRAINING_DATA]

# Stratified train/test split (80/20) — preserves class proportions
X_train, X_test, y_train, y_test = train_test_split(
    queries_all, labels_all,
    test_size=0.2, random_state=42, stratify=labels_all
)

# TF-IDF: unigrams + bigrams, capped at 500 features
scope_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=500)
X_train_vec = scope_vectorizer.fit_transform(X_train)
X_test_vec  = scope_vectorizer.transform(X_test)

# Logistic Regression: interpretable, well-calibrated probabilities
scope_classifier = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
scope_classifier.fit(X_train_vec, y_train)

print("Classifier trained.")
print(f"  Training samples : {len(X_train)}")
print(f"  Test samples     : {len(X_test)}")

### Evaluate the Classifier

Let's check how well it performs on the held-out test set. Because the dataset is small, do not expect perfect numbers: the goal here is to demonstrate the concept (and the XAI transparency that comes with it).

In [ ]:
y_pred = scope_classifier.predict(X_test_vec)

print("Classification Report:\n")
print(classification_report(
    y_test, y_pred,
    target_names=[SCOPE_LABELS[i] for i in range(3)]
))

# Confusion matrix with readable labels
cm = confusion_matrix(y_test, y_pred)
col_width = 16
header = f"{'':20}" + "".join(f"pred:{SCOPE_LABELS[i]:>14}" for i in range(3))
print("Confusion Matrix (rows = actual, columns = predicted):\n")
print(header)
print("-" * len(header))
for i, row in enumerate(cm):
    row_str = f"actual:{SCOPE_LABELS[i]:>13}" + "".join(f"{v:>{col_width}}" for v in row)
    print(row_str)

### XAI: Inspect the Most Influential Features

One of the key advantages of Logistic Regression over a black-box neural classifier is that the model coefficients are directly interpretable. For each class, the features (TF-IDF n-grams) with the highest positive coefficients are the words and phrases that most strongly push the model toward predicting that class.

This lets us audit the classifier: do the top features match our intuition for what "in_scope", "scope_related", and "out_of_scope" mean?

In [ ]:
feature_names = scope_vectorizer.get_feature_names_out()
coef = scope_classifier.coef_   # shape: (n_classes, n_features)

print("Top 10 features per class (highest positive logistic regression coefficients):\n")
for class_idx, class_name in SCOPE_LABELS.items():
    top_idx = np.argsort(coef[class_idx])[-10:][::-1]
    top_features = [(feature_names[j], coef[class_idx][j]) for j in top_idx]
    print(f"  [{class_name}]")
    for feat, weight in top_features:
        print(f"    {feat:<35}  {weight:+.3f}")
    print()

print("Bottom 5 features per class (negative coefficients = pushes away from this class):\n")
for class_idx, class_name in SCOPE_LABELS.items():
    bottom_idx = np.argsort(coef[class_idx])[:5]
    bottom_features = [(feature_names[j], coef[class_idx][j]) for j in bottom_idx]
    print(f"  [{class_name}]")
    for feat, weight in bottom_features:
        print(f"    {feat:<35}  {weight:+.3f}")
    print()

### Integrate the Classifier with the RAG Pipeline

Now we wire the scope classifier as the first gate of the RAG pipeline. The LLM is only called if the classifier allows it, and the scope decision (along with the class probabilities) is always displayed so the user can see exactly why a query was accepted or rejected.

In [ ]:
def classify_scope(query):
    """
    Classify a query using the trained scope classifier.

    Returns
    -------
    label      : int         — 0=in_scope, 1=scope_related, 2=out_of_scope
    label_name : str         — human-readable label
    proba      : np.ndarray  — class probabilities [p_in_scope, p_scope_related, p_out_of_scope]
    """
    vec        = scope_vectorizer.transform([query])
    label      = int(scope_classifier.predict(vec)[0])
    proba      = scope_classifier.predict_proba(vec)[0]
    return label, SCOPE_LABELS[label], proba


def rag_pipeline_with_ml_scope(query):
    """
    Full RAG pipeline with ML-based scope classification as the first gate.

    Flow
    ----
    1. Classify query scope (ML classifier — no LLM call at this stage)
    2a. out_of_scope  → reject immediately; no LLM call is made
    2b. scope_related → run RAG pipeline with a disclaimer
    2c. in_scope      → run RAG pipeline normally
    """
    label, label_name, proba = classify_scope(query)

    proba_str = (
        f"in_scope={proba[0]:.2f}  "
        f"scope_related={proba[1]:.2f}  "
        f"out_of_scope={proba[2]:.2f}"
    )

    print(f"\n{'='*60}")
    print(f"Question: {query}")
    print(f"Scope   : {label_name.upper()}  ({proba_str})")
    print('='*60)

    if label == 2:  # out_of_scope
        print("\nQuery rejected: outside the scope of the knowledge base.")
        print("No LLM call was made.")
        return

    if label == 1:  # scope_related
        print(
            "\nNote: this query is scope-related but may not be fully covered "
            "by the knowledge base. The answer may draw on the LLM's training data."
        )

    retrieved = naive_keyword_search(query, knowledge)
    if retrieved:
        print(f"\nRetrieved {len(retrieved)} document(s):")
        for title, _ in retrieved:
            print(f"  - {title}")
    else:
        print("\nNo relevant documents found in the knowledge base.")
        return

    answer, sources = generate_answer(query, retrieved)
    print(f"\nAnswer:\n{answer}")
    if sources:
        print(f"\nSources: {', '.join(sources)}")
    print()

print("ML-gated RAG pipeline defined.")

Let's run the ML-gated pipeline on a set of representative queries and compare the behaviour to the prompt-based approach from Step 7.

In [ ]:
# in_scope (should retrieve the correct document and answer)
rag_pipeline_with_ml_scope("What is the current population of Berlin?")

In [ ]:
# out_of_scope (should be rejected immediately without calling the LLM)
rag_pipeline_with_ml_scope("How can I install Python on my Windows laptop?")

In [ ]:
# scope_related (about Europe, but not specifically about the 2026 state)
# Should proceed with a disclaimer
rag_pipeline_with_ml_scope("What language do Spanish people speak?")

In [ ]:
# The query that confused the prompt-based guardrail — does the ML classifier handle it correctly?
rag_pipeline_with_ml_scope("Why are film directors starting to shoot movies in Norway?")

In [ ]:
# The prompt-injection attempt — is it rejected by the ML classifier?
rag_pipeline_with_ml_scope("What is the state of quantum computing in Europe and what Python libraries would you recommend for data analysis?")

### Discussion questions

1. **Determinism.** Run the same out-of-scope query five times. Does the ML classifier always produce the same label and the same probabilities? How does this compare to the prompt-based approach from Step 6?

2. **Auditability.** Look at the top features per class printed above. Pick one feature for each class and explain, in plain language, why that word or phrase is a good signal for that class. Does anything surprise you?

3. **Explaining a specific decision.** Run `classify_scope("<your custom query>")` and inspect the probabilities. Now look at which top features overlap with the words in that query. Can you trace *why* the classifier assigned the label it did?

4. **Scope-related handling.** When the classifier returns `scope_related`, the pipeline proceeds with a disclaimer. Is this the right policy? What are the risks of answering scope-related questions at all? What alternative policies could you implement?

5. **Dataset quality.** The classifier was trained on only ~90 hand-crafted examples. Where do you see it fail or give low-confidence predictions? What categories of queries would you add to make it more robust?

6. **Trade-offs.** The ML classifier requires a labelled dataset and must be retrained whenever the scope definition changes. In what situations would you prefer the prompt-based approach? In what situations is the ML classifier worth the extra effort?